# Playground for Marint Naturkart hard/soft bottom

- Loads NGU sediment data and supporting region layers (municipalities and sea region)
- Classifies sediment polygons into BunnType categories: løsbunn, fastbunn, or uspesifisert
- Clips data to the sea area within selected municipalities
- Identifies areas missing bottom classification by differencing sea coverage and sediment union
- Appends missing areas as placeholder rows with BunnType="missing"
- Some of the datasets can be explored on the test terriamap [here](https://terriamap.t.niva.no/#start=%7B%22version%22%3A%228.0.0%22%2C%22initSources%22%3A%5B%7B%22stratum%22%3A%22user%22%2C%22models%22%3A%7B%22%2F%2FNGU+-+MarinBunnsedimenterWMS%22%3A%7B%22isOpen%22%3Atrue%2C%22knownContainerUniqueIds%22%3A%5B%22%2F%22%5D%2C%22type%22%3A%22group%22%7D%2C%22ngu_marinbunnsedimenter%22%3A%7B%22isOpen%22%3Atrue%2C%22knownContainerUniqueIds%22%3A%5B%22%2F%2FNGU+-+MarinBunnsedimenterWMS%22%5D%2C%22type%22%3A%22wms-group%22%7D%2C%22%2F%2FMarint+Natur+Kart%2FBunntyper+Union+AOI+%22%3A%7B%22show%22%3Atrue%2C%22opacity%22%3A1%2C%22activeStyle%22%3A%22BunnType%22%2C%22knownContainerUniqueIds%22%3A%5B%22%2F%2FMarint+Natur+Kart%22%5D%2C%22type%22%3A%22geojson%22%7D%2C%22ngu_marinbunnsedimenter%2FMarinBunnsedimenterWMS%2FSedimentkornstorrelse%2FKornstorrelse_Det%22%3A%7B%22show%22%3Atrue%2C%22isOpenInWorkbench%22%3Afalse%2C%22opacity%22%3A0.35%2C%22knownContainerUniqueIds%22%3A%5B%22ngu_marinbunnsedimenter%2FMarinBunnsedimenterWMS%2FSedimentkornstorrelse%22%5D%2C%22type%22%3A%22wms%22%7D%2C%22ngu_marinbunnsedimenter%2FMarinBunnsedimenterWMS%2FSedimentkornstorrelse%22%3A%7B%22isOpen%22%3Atrue%2C%22knownContainerUniqueIds%22%3A%5B%22ngu_marinbunnsedimenter%2FMarinBunnsedimenterWMS%22%5D%2C%22type%22%3A%22group%22%7D%2C%22ngu_marinbunnsedimenter%2FMarinBunnsedimenterWMS%22%3A%7B%22isOpen%22%3Atrue%2C%22knownContainerUniqueIds%22%3A%5B%22ngu_marinbunnsedimenter%22%5D%2C%22type%22%3A%22group%22%7D%2C%22%2F%22%3A%7B%22type%22%3A%22group%22%7D%2C%22%2F%2FMarint+Natur+Kart%22%3A%7B%22knownContainerUniqueIds%22%3A%5B%22%2F%22%5D%2C%22type%22%3A%22group%22%7D%7D%2C%22workbench%22%3A%5B%22ngu_marinbunnsedimenter%2FMarinBunnsedimenterWMS%2FSedimentkornstorrelse%2FKornstorrelse_Det%22%2C%22%2F%2FMarint+Natur+Kart%2FBunntyper+Union+AOI+%22%5D%2C%22timeline%22%3A%5B%22ngu_marinbunnsedimenter%2FMarinBunnsedimenterWMS%2FSedimentkornstorrelse%2FKornstorrelse_Det%22%2C%22%2F%2FMarint+Natur+Kart%2FBunntyper+Union+AOI+%22%5D%2C%22initialCamera%22%3A%7B%22west%22%3A5.338282585144044%2C%22south%22%3A62.19159014075552%2C%22east%22%3A5.37830114364624%2C%22north%22%3A62.20154880517489%7D%2C%22homeCamera%22%3A%7B%22west%22%3A4%2C%22south%22%3A57.00000000000001%2C%22east%22%3A32%2C%22north%22%3A72%7D%2C%22viewerMode%22%3A%222d%22%2C%22showSplitter%22%3Afalse%2C%22splitPosition%22%3A0.5%2C%22settings%22%3A%7B%22baseMaximumScreenSpaceError%22%3A2%2C%22useNativeResolution%22%3Afalse%2C%22alwaysShowTimeline%22%3Afalse%2C%22baseMapId%22%3A%22basemap-openstreetmap%22%2C%22terrainSplitDirection%22%3A0%2C%22depthTestAgainstTerrainEnabled%22%3Afalse%7D%2C%22stories%22%3A%5B%5D%7D%5D%7D) there can be breaking changes for the link.

To make the code easy to run it reads data from a public accessible cloud storage bucket [gs://niva-geodata](https://console.cloud.google.com/storage/browser/niva-geodata/MarintNaturKart;tab=objects?project=nivaprod-1&prefix=&forceOnObjectsSortingFiltering=false) these dataset are work in progress.

In [2]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
import utils

In [3]:
def save_layer_as_geoparquet(input_gml_path: Path, layer_name: str, output_path: Path):
    """Save GML Download from NGU as GeoParquet

    Geoparquet is a really nice format for geospatial data optimized for cloud access.
    
    For the data also see https://geo.ngu.no/download/ or https://niva365.sharepoint.com/sites/MarintNaturkart/Shared%20Documents/Forms/AllItems.aspx?CT=1764100542344&OR=OWA%2DNT%2DMail&CID=f1061c4b%2D460d%2D43bf%2Db385%2Da42f42a569c8&csf=1&web=1&e=8XCuTS&FolderCTID=0x012000289EDE01C0CE7544AEDB28C61E3A7933&id=%2Fsites%2FMarintNaturkart%2FShared%20Documents%2FGeneral%2F03%20Data%2FNisjedata%2Fngu%5Fsediment
    if you have access.
    """
    gdf = gpd.read_file(input_gml_path, layer=layer_name)
    gdf.to_parquet(output_path, compression="snappy")
    print(f"Written GeoParquet to {output_path}")


# Only needed once
# Downloaded from https://kartkatalog.geonorge.no/metadata/bunnsedimenter-kornstoerrelse-detaljert/79f0f17d-9f62-456d-b1a9-2a8c754c51c4
sediments_detailed = Path(
    "../ngu_sediment/Geologi_0000_Norge_25833_BunnsedimentKornstorDetalj_GML/BunnsedimentKornstorDetalj.gml"
)
output_path = Path("ngu_sediment/BunnsedimentKornstorDetalj.geo.parquet")
# save_layer_as_geoparquet(sediments_detailed, "KornstrFlate", output_path)

# Downloaded from https://dataleveranser.miljodirektoratet.no/nedlasting/b0fa9d93-dd2d-48fd-a89f-8fa224555c4d/
marine_vanntyper = Path(
    "../mdir/NyTypologi2022.shp"
)
output_path = Path("../mdir/NyTypologi2022.geo.parquet")
#save_layer_as_geoparquet(marine_vanntyper, None, output_path)

# Setup

Initial setup for area of interest and mapping of NGU [bunnsedimenter](https://kartkatalog.geonorge.no/metadata/79f0f17d-9f62-456d-b1a9-2a8c754c51c4).

In [52]:
# https://static.ngu.no/Mareano/Kornstorrelse.html
# also see https://static.ngu.no/Mareano/Kornstorrelse-NiN-LM.html  DK_EFGY hard while, DK_ABCD soft and DK_0 mixture
HARD_BOTTOM_LM_DK = ["DK_EFGY"]
SOFT_BOTTOM_LM_DK = ["DK_AB", "DK_C", "DK_D"]
MIXED_BOTTOM_LM_DK = ["DK_0", "0"]
KOMMUNER = {
    "1515": "Herøy",
    "1516": "Ulstein",
    "1520": "Ørsta",
    "1577": "Volda",
    "1508": "Ålesund",
    "1532": "Giske",
    "1514": "Sande",
    "1531": "Sula",
    "1580": "Haram",
    "1528": "Sykkylven",
    "1517": "Hareid",
    "1511": "Vanylven",
}

In [ ]:
df_kornstr = pd.read_html("https://static.ngu.no/Mareano/Kornstorrelse-NiN-LM.html", skiprows=0)[1]

,Kornstørrelse SOSI-kode,Kornstørrelse SOSI-navn,Dominerende kornstørrelse LM-DK,Finmaterialinnhold LM-FI
0,10,Leir,DK_AB,FI_y
1,15,Organisk slam,DK_AB,FI_d
2,20,Slam,DK_AB,FI_y
3,21,Slam med blokker av sedimenter,DK_AB,FI_d
4,30,Sandholdig leir,DK_AB,FI_d
5,40,Sandholdig slam,DK_AB,FI_d
6,50,Silt,DK_AB,FI_y
7,60,Sandholdig silt,DK_AB,FI_d
8,70,Leirholdig sand,DK_C,FI_c
9,80,Slamholdig sand,DK_C,FI_c


In [58]:
SOFT_BOTTOM_TYPES = {
    row["Kornstørrelse SOSI-kode"]: row["Kornstørrelse SOSI-navn"]
    for _, row in df_kornstr[
        df_kornstr["Dominerende kornstørrelse LM-DK"].isin(SOFT_BOTTOM_LM_DK)
    ].iterrows()
}
HARD_BOTTOM_TYPES = {
    row["Kornstørrelse SOSI-kode"]: row["Kornstørrelse SOSI-navn"]
    for _, row in df_kornstr[
        df_kornstr["Dominerende kornstørrelse LM-DK"].isin(HARD_BOTTOM_LM_DK)
    ].iterrows()
}
MIXTURE_BOTTOM_TYPES = {
    row["Kornstørrelse SOSI-kode"]: row["Kornstørrelse SOSI-navn"]
    for _, row in df_kornstr[
        df_kornstr["Dominerende kornstørrelse LM-DK"].isin(MIXED_BOTTOM_LM_DK)
    ].iterrows()
}
# Build a table with multi-level columns: main headers (hard, soft, mix) and sub headers (key, value)
dicts = {
    "hard": HARD_BOTTOM_TYPES,
    "soft": SOFT_BOTTOM_TYPES,
    "mix": MIXTURE_BOTTOM_TYPES,
}

df_types = pd.concat(
    {
        name: pd.Series(d).rename_axis("key").reset_index().rename(columns={0: "value"})
        for name, d in dicts.items()
    },
    axis=1,
)
df_types.columns = pd.MultiIndex.from_tuples(df_types.columns)
df_types

hard                                                    soft  \
      key                                              value  key   
0   180.0                                     Stein og blokk   10   
1   300.0       Harde sedimenter eller sedimentære bergarter   15   
2     1.0  Tynt eller usammenhengende sedimentdekke over ...   20   
3     5.0                                         Bart fjell   21   
4     NaN                                                NaN   30   
5     NaN                                                NaN   40   
6     NaN                                                NaN   50   
7     NaN                                                NaN   60   
8     NaN                                                NaN   70   
9     NaN                                                NaN   80   
10    NaN                                                NaN   90   
11    NaN                                                NaN   95   
12    NaN                                                NaN  100   
13    NaN                                                NaN  105   
14    NaN                                                NaN  110   
15    NaN                                                NaN  115   
16    NaN                                                NaN  120   
17    NaN                                                NaN  130   
18    NaN                                                NaN  140   
19    NaN                                                NaN  150   
20    NaN                                                NaN  160   
21    NaN                                                NaN  170   
22    NaN                                                NaN  174   
23    NaN                                                NaN  175   

                                      mix  \
                             value    key   
0                             Leir  185.0   
1                    Organisk slam  190.0   
2                             Slam  205.0   
3   Slam med blokker av sedimenter  206.0   
4                  Sandholdig leir  210.0   
5                  Sandholdig slam  215.0   
6                             Silt  500.0   
7                  Sandholdig silt    0.0   
8                  Leirholdig sand    NaN   
9                  Slamholdig sand    NaN   
10                 Siltholdig sand    NaN   
11                        Fin sand    NaN   
12                            Sand    NaN   
13                       Grov sand    NaN   
14                 Grusholdig slam    NaN   
15      Grusholdig sandholdig slam    NaN   
16      Grusholdig slamholdig sand    NaN   
17                 Grusholdig sand    NaN   
18                 Slamholdig grus    NaN   
19      Slamholdig sandholdig grus    NaN   
20                 Sandholdig grus    NaN   
21                            Grus    NaN   
22                   Grus og stein    NaN   
23            Grus, stein og blokk    NaN   

                                              
                                       value  
0                        Sand, grus og stein  
1                              Sand og blokk  
2                  Slam/sand med stein/blokk  
3           Slam, sand, grus, stein og blokk  
4            Stein/blokk med sand-/slamdekke  
5                 Sand, grus, stein og blokk  
6      Biogent materiale (korall, korallrev)  
7   Uspesifisert med hensyn på kornstørrelse  
8                                        NaN  
9                                        NaN  
10                                       NaN  
11                                       NaN  
12                                       NaN  
13                                       NaN  
14                                       NaN  
15                                       NaN  
16                                       NaN  
17                                       NaN  
18                                       NaN  
19                                       NaN  
20   

In [36]:
gdf_bunn = gpd.read_parquet(
    "gs://niva-geodata/MarintNaturKart/ngu_sediment/BunnsedimentKornstorDetalj.geo.parquet"
)

In [59]:
def to_bunn_type(kornstorrelse: int) -> str:
    if kornstorrelse in SOFT_BOTTOM_TYPES:
        return "løsbunn"
    elif kornstorrelse in HARD_BOTTOM_TYPES:
        return "fastbunn"
    else:
        return "blanding/uspesifisert"



In [60]:
gdf_bunn["BunnType"] = gdf_bunn["sedKornstørrelse"].map(to_bunn_type)

In [69]:
fname = "soft_hard_bunn_from_BunnsedimentKornstorDetalj"
gdf_bunn.to_file(f"{fname}.geojson", driver="GeoJSON")
gdf_bunn.to_parquet(f"{fname}.geo.parquet", compression="snappy")

## Clip kommuner to the sea region geometry

The coast line is created by fieldGeo

In [62]:
gdf_kommuner = gpd.read_file(
    "https://storage.googleapis.com/niva-geodata/MarintNaturKart/kommuner_simplified.geojson"
).query("kommunenummer in @KOMMUNER")


In [63]:
sea_region = gpd.read_file("https://storage.googleapis.com/niva-geodata/MarintNaturKart/sea_union-from-HI-moere_og_romsdal.gpkg")

In [64]:
# Ensure matching CRS before clipping
if gdf_kommuner.crs != sea_region.crs:
    sea_region = sea_region.to_crs(gdf_kommuner.crs)

gdf_kommuner_sea = gpd.clip(gdf_kommuner, sea_region)

In [65]:
# Select polygons from gdf_bunn that are inside gdf_kommuner_sea
if gdf_bunn.crs != gdf_kommuner_sea.crs:
    gdf_bunn = gdf_bunn.to_crs(gdf_kommuner_sea.crs)

kommuner_sea_union = gdf_kommuner_sea.geometry.union_all()
gdf_final = gdf_bunn[gdf_bunn.geometry.within(kommuner_sea_union)].copy()

print(f"Selected {len(gdf_final)} of {len(gdf_bunn)} polygons inside kommuner sea mask.")

Selected 14029 of 130953 polygons inside kommuner sea mask.


## Create ploygons missing classification

Set all columns to 'missing' initally

In [70]:
bunn_union = gdf_final.geometry.union_all()

missing_geom = kommuner_sea_union.difference(bunn_union)

gdf_missing_bunn_union = gpd.GeoDataFrame(geometry=[missing_geom], crs=gdf_kommuner_sea.crs)

missing_parts = gdf_missing_bunn_union.explode(index=False, ignore_index=True)

cols = [c for c in gdf_final.columns if c != "geometry"]
missing_df = {c: ["missing"] * len(missing_parts) for c in cols}

gdf_missing_rows = gpd.GeoDataFrame(missing_df, geometry=missing_parts.geometry, crs=gdf_final.crs)

# Append to existing selection
gdf_final = gpd.GeoDataFrame(
    pd.concat([gdf_final, gdf_missing_rows], ignore_index=True),
    geometry="geometry",
    crs=gdf_final.crs,
)

In [71]:
# Create GeoDataFrame of missing areas with BunnType="missing" and other fields None,
# then append to gdf_bunn_in_kommuner_sea
missing_parts = gdf_missing_bunn_union.explode(index=False, ignore_index=True)

# Build a DataFrame with same columns as gdf_bunn_in_kommuner_sea
cols = [c for c in gdf_final.columns if c != "geometry"]
missing_df = {c: ['missing'] * len(missing_parts) for c in cols}
missing_df["BunnType"] = ["missing"] * len(missing_parts)

gdf_missing_rows = gpd.GeoDataFrame(missing_df, geometry=missing_parts.geometry, crs=gdf_final.crs)

gdf_final = gpd.GeoDataFrame(
    pd.concat([gdf_final, gdf_missing_rows], ignore_index=True),
    geometry="geometry",
    crs=gdf_final.crs,
)


## Store inital labels

Store inital labels based on NGU data and missing area to reusable datasets. The geojson can be loaded into terriamap for comparison when exploring features, datasets and models more. 

In [73]:
fname = utils.to_filename("nisjedata-substrat-klassifisering", "moere-og-romsdal", "latest", gdf_final.crs.to_epsg())
gdf_final.to_file(f"{fname}.geojson", driver="GeoJSON")
gdf_final.to_file(f"{fname}.gpkg", layer="soft_hard_bottom", driver="GPKG")